In [1]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

processed_path = project_path / "data" / "processed"

train_df = pd.read_csv(processed_path / "train.csv")
val_df = pd.read_csv(processed_path / "validation.csv")
test_df = pd.read_csv(processed_path / "test.csv")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

display(train_df[["label", "text"]].head())

Training: (803, 3)
Validation: (178, 3)
Testing: (172, 3)


,label,text
0,Psoriasis,I have been experiencing a skin rash on my arm...
1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,Psoriasis,I have been experiencing joint pain in my fing...
3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,Psoriasis,"My nails have small dents or pits in them, and..."


In [2]:
import re

replacements = [
    (r"\bI have been experiencing\b", "I have"),
    (r"\bI am experiencing\b", "I have"),
    (r"\bI've been experiencing\b", "I have"),
]

def augment_text(text):
    for pattern, replacement in replacements:
        new_text = re.sub(
            pattern,
            replacement,
            text,
            count=1,
            flags=re.IGNORECASE
        )

        if new_text != text:
            return new_text

    return None

# Testojme funksionin
original = "I have been experiencing headaches and nausea."

augmented = augment_text(original)

print("Original:", original)
print("Augmented:", augmented)

Original: I have been experiencing headaches and nausea.
Augmented: I have headaches and nausea.


In [3]:
augmented_rows = []

for _, row in train_df.iterrows():

    original_text = row["text"]
    new_text = augment_text(original_text)

    if new_text is not None:
        augmented_rows.append({
            "label": row["label"],
            "original_text": original_text,
            "text": new_text,
            "group_id": row["group_id"],
            "method": "controlled_replacement"
        })

augmented_df = pd.DataFrame(
    augmented_rows,
    columns=[
        "label", "original_text", "text",
        "group_id", "method"
    ]
)

print("Original training rows:", len(train_df))
print("Generated candidates:", len(augmented_df))

display(augmented_df.head(10))

Original training rows: 803
Generated candidates: 92


,label,original_text,text,group_id,method
0,Psoriasis,I have been experiencing a skin rash on my arm...,"I have a skin rash on my arms, legs, and torso...",0,controlled_replacement
1,Psoriasis,I have been experiencing joint pain in my fing...,"I have joint pain in my fingers, wrists, and k...",2,controlled_replacement
2,Psoriasis,"My skin is prone to infections due to dry, fla...","My skin is prone to infections due to dry, fla...",14,controlled_replacement
3,Varicose Veins,I have noticed that the blood vessels in my le...,I have noticed that the blood vessels in my le...,94,controlled_replacement
4,Varicose Veins,I am experiencing too many cramps in the last ...,I have too many cramps in the last couple of d...,98,controlled_replacement
5,Typhoid,"I have been experiencing chills and fever, alo...","I have chills and fever, along with severe abd...",102,controlled_replacement
6,Typhoid,"I've been experiencing high fever, especially ...","I have high fever, especially at night. It's b...",107,controlled_replacement
7,Typhoid,I am experiencing a lot of belly pain and cons...,"I have a lot of belly pain and constipation, w...",113,controlled_replacement
8,Typhoid,I have been experiencing a lot of bloating and...,"I have a lot of bloating and constipation, and...",115,controlled_replacement
9,Typhoid,I am experiencing extreme belly pain and const...,I have extreme belly pain and constipation. Ev...,116,controlled_replacement


In [4]:
if augmented_df.empty:
    print("No augmentation candidates generated.")
else:
    existing_train = set(train_df["text"])
    existing_val = set(val_df["text"])
    existing_test = set(test_df["text"])

    print("Generated:", len(augmented_df))

    print("Already in Training:",
          augmented_df["text"].isin(existing_train).sum())

    print("Already in Validation:",
          augmented_df["text"].isin(existing_val).sum())

    print("Already in Testing:",
          augmented_df["text"].isin(existing_test).sum())

    print("Duplicate generated texts:",
          augmented_df["text"].duplicated().sum())

    display(
        augmented_df[
            ["label", "original_text", "text"]
        ].head(10)
    )

Generated: 92
Already in Training: 0
Already in Validation: 0
Already in Testing: 0
Duplicate generated texts: 0


,label,original_text,text
0,Psoriasis,I have been experiencing a skin rash on my arm...,"I have a skin rash on my arms, legs, and torso..."
1,Psoriasis,I have been experiencing joint pain in my fing...,"I have joint pain in my fingers, wrists, and k..."
2,Psoriasis,"My skin is prone to infections due to dry, fla...","My skin is prone to infections due to dry, fla..."
3,Varicose Veins,I have noticed that the blood vessels in my le...,I have noticed that the blood vessels in my le...
4,Varicose Veins,I am experiencing too many cramps in the last ...,I have too many cramps in the last couple of d...
5,Typhoid,"I have been experiencing chills and fever, alo...","I have chills and fever, along with severe abd..."
6,Typhoid,"I've been experiencing high fever, especially ...","I have high fever, especially at night. It's b..."
7,Typhoid,I am experiencing a lot of belly pain and cons...,"I have a lot of belly pain and constipation, w..."
8,Typhoid,I have been experiencing a lot of bloating and...,"I have a lot of bloating and constipation, and..."
9,Typhoid,I am experiencing extreme belly pain and const...,I have extreme belly pain and constipation. Ev...


In [5]:
print("Total generated:", len(augmented_df))

unchanged = (
    augmented_df["original_text"]
    == augmented_df["text"]
).sum()

print("Unchanged texts:", unchanged)

print("\nFirst 5 full examples:")

for _, row in augmented_df.head(5).iterrows():
    print("\nDisease:", row["label"])
    print("ORIGINAL:", row["original_text"])
    print("AUGMENTED:", row["text"])
    print("-" * 60)

Total generated: 92
Unchanged texts: 0

First 5 full examples:

Disease: Psoriasis
ORIGINAL: I have been experiencing a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches.
AUGMENTED: I have a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches.
------------------------------------------------------------

Disease: Psoriasis
ORIGINAL: I have been experiencing joint pain in my fingers, wrists, and knees. The pain is often achy and throbbing, and it gets worse when I move my joints.
AUGMENTED: I have joint pain in my fingers, wrists, and knees. The pain is often achy and throbbing, and it gets worse when I move my joints.
------------------------------------------------------------

Disease: Psoriasis
ORIGINAL: My skin is prone to infections due to dry, flaky patches. I am experiencing a strong pain in my joints. The skin on my knees and elbows is starting to peel o

In [6]:
distribution = pd.DataFrame({
    "Original": train_df["label"].value_counts(),
    "Augmented": augmented_df["label"].value_counts()
}).fillna(0).astype(int)

distribution["Total"] = (
    distribution["Original"]
    + distribution["Augmented"]
)

display(distribution.sort_index())

print("\nTotal original:", distribution["Original"].sum())
print("Total generated:", distribution["Augmented"].sum())

,Original,Augmented,Total
label,,,
Acne,34,1,35
Arthritis,29,6,35
Bronchial Asthma,33,1,34
Cervical spondylosis,34,11,45
Chicken pox,33,2,35
Common Cold,31,3,34
Dengue,33,8,41
Dimorphic Hemorrhoids,26,5,31
Fungal infection,39,2,41



Total original: 803
Total generated: 92


In [7]:
output_file = processed_path / "augmentation_candidates.csv"

augmented_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_file)
print("Candidates:", len(augmented_df))

Saved: c:\Projects\Bachelor-Thesis\data\processed\augmentation_candidates.csv
Candidates: 92


In [8]:
from difflib import SequenceMatcher

augmented_df["similarity_to_original"] = augmented_df.apply(
    lambda row: SequenceMatcher(
        None,
        row["original_text"].lower(),
        row["text"].lower()
    ).ratio(),
    axis=1
)

print(
    augmented_df["similarity_to_original"].describe()
)

display(
    augmented_df[
        ["label", "original_text", "text",
         "similarity_to_original"]
    ].sort_values(
        "similarity_to_original",
        ascending=False
    ).head(10)
)

count    92.000000
mean      0.886395
std       0.199657
min       0.044643
25%       0.925795
50%       0.940885
75%       0.947796
max       0.962594
Name: similarity_to_original, dtype: float64


,label,original_text,text,similarity_to_original
28,Common Cold,"My neck is swollen, and I'm exhausted. My thro...","My neck is swollen, and I'm exhausted. My thro...",0.962594
22,Dengue,I am experiencing very high fever and chills e...,I have very high fever and chills every night....,0.959660
7,Typhoid,I am experiencing a lot of belly pain and cons...,"I have a lot of belly pain and constipation, w...",0.957507
9,Typhoid,I am experiencing extreme belly pain and const...,I have extreme belly pain and constipation. Ev...,0.956268
10,Typhoid,I am experiencing a lot of nausea and vomiting...,"I have a lot of nausea and vomiting, and it's ...",0.956012
76,Jaundice,"I've been feeling extremely scratchy, sick, an...","I've been feeling extremely scratchy, sick, an...",0.955882
15,Impetigo,I am experiencing skin rashes with burning sor...,I have skin rashes with burning sores. The bli...,0.955490
57,Migraine,"I have been experiencing digestive issues, inc...","I have digestive issues, including acidity and...",0.955000
4,Varicose Veins,I am experiencing too many cramps in the last ...,I have too many cramps in the last couple of d...,0.954128
71,Jaundice,"I have been experiencing intense itching, vomi...","I have intense itching, vomiting, and fatigue....",0.954082


In [9]:
low_similarity = augmented_df[
    augmented_df["similarity_to_original"] < 0.70
]

print("Low similarity candidates:", len(low_similarity))

for _, row in low_similarity.iterrows():
    print("\nDisease:", row["label"])
    print("Similarity:", round(row["similarity_to_original"], 4))
    print("ORIGINAL:", row["original_text"])
    print("AUGMENTED:", row["text"])
    print("-" * 60)

Low similarity candidates: 6

Disease: Common Cold
Similarity: 0.4642
ORIGINAL: I've been coughing a lot and feeling chilly and shivery. My nose has been quite clogged, and I am experiencing facial pressure. I also have a lot of phlegm, and coughing pains my chest. I can't smell anything, and my muscles are aching.
AUGMENTED: I've been coughing a lot and feeling chilly and shivery. My nose has been quite clogged, and I have facial pressure. I also have a lot of phlegm, and coughing pains my chest. I can't smell anything, and my muscles are aching.
------------------------------------------------------------

Disease: Pneumonia
Similarity: 0.0446
ORIGINAL: I've been experiencing chills, feeling worn out, and I can't seem to get rid of this cough. When I cough, my chest aches and my heart feels as like it is beating a million miles per hour. I'm coughing up a nasty rust-colored phlegm.
AUGMENTED: I have chills, feeling worn out, and I can't seem to get rid of this cough. When I cough, my